# INITIAL IMPORT

In [2]:
%load_ext autoreload
%autoreload 2

import numpy as np
import pandas as pd
pd.set_option('display.max_rows', 500)
pd.set_option('display.max_columns', 500)

In [3]:
from src.config import Configuration
from src.tetris import TetrisConfiguration

T_CONFIG = TetrisConfiguration(
)

CONFIG = Configuration(
)

# Data

### Rotation 'r'
- N: north, AKA "0"
- E: east, AKA "R"
- S: south, AKA "2"
- W: west, AKA "L"

### Playfield
- N: Empty
- G: Garbage

### Won and ranking
In case we want to use only won games. Max ranking25000

In [128]:
raw_df = pd.read_csv(CONFIG.raw_dataset_path)
print(raw_df.columns)

raw_df = raw_df.sort_values(by=['game_id', 'subframe'])
raw_df['subframe'] = raw_df.groupby('game_id').cumcount()
raw_df['playfield'] = raw_df['playfield'].fillna('N')
raw_df['playfield_next'] = raw_df['playfield'].shift(-1).fillna('N')
# raw_df['playfield'].replace('NaN', 'N', inplace=True)
mask_keep_all_but_last = raw_df.duplicated(subset=['game_id'], keep='last')
raw_df = raw_df[mask_keep_all_but_last].copy()

# Fix: data pipeline systematically swaps J and L piece labels
# (coordinate mirror in the TETR.IO extractor)
swap_jl = {'J': 'L', 'L': 'J'}
raw_df['placed'] = raw_df['placed'].replace(swap_jl)
raw_df['hold'] = raw_df['hold'].replace(swap_jl)

# 'placed' is always the piece that was active and placed.
# 'hold' is always the current hold slot value.
# No derivation needed — these are directly the game state variables.
raw_df['real_current'] = raw_df['placed']
raw_df['real_hold'] = raw_df['hold']
raw_df = raw_df[raw_df['subframe'] > 0].copy()

# If lines were cleared, no garbage
raw_df['immediate_garbage'] = np.where(raw_df['cleared'] > 0, 0, raw_df['immediate_garbage'])
# Cap at 8, as that's the max garbage that can be received in one frame
raw_df['immediate_garbage'] = raw_df['immediate_garbage'].clip(upper=8)

# raw_df = raw_df.groupby('game_id')
columns = ['game_id', 'subframe', 'playfield', 'playfield_next', 'real_current', 'placed', 'real_hold', 'hold', 'next', 'won', 'rating', 'immediate_garbage', 'btb', 'combo']
raw_df[columns].to_csv(CONFIG.processed_dataset_path, index=False)

Index(['game_id', 'subframe', 'won', 'playfield', 'x', 'y', 'r', 'placed',
       'hold', 'next', 'cleared', 'garbage_cleared', 'attack', 't_spin', 'btb',
       'combo', 'immediate_garbage', 'incoming_garbage', 'rating', 'glicko',
       'glicko_rd'],
      dtype='str')


In [129]:
raw_df[columns].sample(10)

,game_id,subframe,playfield,playfield_next,real_current,placed,real_hold,hold,next,won,rating,immediate_garbage,btb,combo
1235996,23266,91,IIIINOOJJJLLLNNNNNNNL,IIIINOOJJJLLLNNNNLLLLNNNNNNL,J,J,T,T,OIJSZZILJOTSTL,0,24854.812500,0,1,0
3380760,58079,52,GGGGGGGNGGGGGGGGGNGGGGGGGGGNGGGGGGGGGNGGGGGGGG...,GGGGGGGNGGGGGGGGGNGGGGGGGGGNGGGGGGGGGNGGGGGGGG...,T,T,O,O,SJLTJSIZOSLZIJ,1,24951.585938,0,0,0
3534443,60519,116,GGNGGGGGGGSSZZNSSLLLSZZLNSTJJJJLLLNTTTZJJJJNNN...,GGNGGGGGGGSSZZNSSLLLSZZLNSTJJJJJJSSNOOZZJJJSNN...,S,S,J,J,TTIZSLJOZJSOIL,1,24813.966797,0,2,2
6450991,106780,1,NNSSNNNNNNNNNSS,NNSSNNNNNINNNSSNNNNINNNNNNNNNINNNNNNNNNI,I,I,Z,Z,OTLJLTSIJZOTIS,0,24989.593750,0,0,0
2508427,44707,56,NGGGGGGGGGNGGGGGGGGGNGGGGGGGGGNGGGGGGGGGNGGGGG...,NGGGGGGGGGNGGGGGGGGGNGGGGGGGGGNGGGGGGGGGNGGGGG...,L,L,O,O,ZILSTLZSOTJIIJ,1,24984.250000,0,1,1
6241378,103129,11,LLLNNNNNNNL,LLLNNNIIIIL,I,I,O,O,ZIZOTJLSZOTISJ,1,24909.371094,0,0,0
1671695,30877,171,GGGGNGGGGGGGGGNGGGGGGGGGGGGGGNGGGGGGGGGNGGGGGG...,GGGGNGGGGGGGGGNGGGGGLOOZZZJLSNTOOZZZJLLNTTOOZJ...,I,I,S,S,ILOITJZLSILZTO,1,24964.166016,0,0,0
7628862,123032,18,GGGGGGGGNGGGGGGGGGNGJJJNNLSOOIJJNNNLLOOIJNNNNI...,GGGGGGGGNGGGGGGGGGNGJJNOOLLOOIJNNNNIIIIIJ,O,O,Z,Z,SSZIOJTLTISZJL,0,24967.193359,0,3,0
2746600,48252,27,GGGGGGGGGNGGGGGGGGGNGGGGGGGGGNGGGGGGGGGNGGGGGG...,GGGGGGGGNGGGGNGGGGGGGGGGNGGGGGGGGGNGGGGGGGGGGG...,L,L,O,O,ZTJLISLZOJSTIO,0,24820.375000,4,0,1
6707083,110633,192,GGGGGGGGNGGGGGGGGGNGGGGGNGGGGGLLOOSSNIOOILJLSS...,GGGGGGGGNGGGGGGGGGNGGGGGNGGGGGLLOOSSNIOOILJLSS...,L,L,I,I,TZZJLOTISOIZLT,1,24980.460938,0,5,0


In [130]:
i = 9

In [131]:
print(raw_df[(raw_df['game_id'] == 18) & (raw_df['subframe'] == i)]['playfield'].values[0])
print(raw_df[(raw_df['game_id'] == 18) & (raw_df['subframe'] == i)]['playfield_next'].values[0])
print(raw_df[(raw_df['game_id'] == 18) & (raw_df['subframe'] == i)]['immediate_garbage'].values[0])

i += 1

GGNGGGGGGGGGNGGGGGGGJJJOONNTLLJJJOONNTTLJLZIIIITNLJLZZNSSNNNNLLZNNSS
GGNGGGGGGGGGNGGGGGGGJJJOONNTLLJJJOONNTTLJLZIIIITNLJLZZZSSNNNNLLZZZSSNNNNNNNZ
0


In [132]:
raw_df[(raw_df['game_id'] == 18) & ((raw_df['subframe'] == 10) | (raw_df['subframe'] == 9) | (raw_df['subframe'] == 11))][columns]

,game_id,subframe,playfield,playfield_next,real_current,placed,real_hold,hold,next,won,rating,immediate_garbage,btb,combo
1671,18,9,GGNGGGGGGGGGNGGGGGGGJJJOONNTLLJJJOONNTTLJLZIII...,GGNGGGGGGGGGNGGGGGGGJJJOONNTLLJJJOONNTTLJLZIII...,Z,Z,S,S,TIOSZOTJLIJSTZ,0,24748.521484,0,0,0
1672,18,10,GGNGGGGGGGGGNGGGGGGGJJJOONNTLLJJJOONNTTLJLZIII...,GGNGGGGGGGGGNGGGGGGGJJJOONNTLLJJJOONNTTLNLLZZZ...,T,T,S,S,IOSZOTJLIJSTZL,0,24748.521484,0,0,0
1673,18,11,GGNGGGGGGGGGNGGGGGGGJJJOONNTLLJJJOONNTTLNLLZZZ...,GGNGGGGGGGGGNGGGGGGGGGNGGGGGGGJJJOONNTLLJJJOON...,I,I,S,S,OSZOTJLIJSTZLI,0,24748.521484,1,1,1


### Split

In [133]:
print('Loading processed dataset...')
processed_df = pd.read_csv(CONFIG.processed_dataset_path)

print('Other stuff...')
# Shuffle by game_id to prevent data leakage (keeping frames of the same game together)
unique_games = pd.Series(processed_df['game_id'].unique()).sample(frac=1, random_state=CONFIG.seed)

# Calculate split indices assuming CONFIG sizes are floats (e.g., 0.1 for 10%)
train_end = int(len(unique_games) * (1 - CONFIG.test_size - CONFIG.val_size))
val_end = train_end + int(len(unique_games) * CONFIG.val_size)

# Partition the shuffled game IDs
train_ids = unique_games.iloc[:train_end]
val_ids = unique_games.iloc[train_end:val_end]
test_ids = unique_games.iloc[val_end:]

# Create the final partitions
train_df = processed_df[processed_df['game_id'].isin(train_ids)].copy()
train_df = train_df.sample(frac=1, random_state=CONFIG.seed).reset_index(drop=True)
val_df = processed_df[processed_df['game_id'].isin(val_ids)].copy()
test_df = processed_df[processed_df['game_id'].isin(test_ids)].copy()

print(f"Train games: {len(train_ids):_}, Val games: {len(val_ids):_}, Test games: {len(test_ids):_}")
print('Saving partitions...')
train_df.to_csv(CONFIG.tetrio_train, index=False)
val_df.to_csv(CONFIG.tetrio_val, index=False)
test_df.to_csv(CONFIG.tetrio_test, index=False)

Loading processed dataset...
Other stuff...
Train games: 53_683, Val games: 7_669, Test games: 15_338
Saving partitions...


### Show dataloaders

In [4]:
from src.data import load_tetrio_data

train_loader, test_loader, val_loader = load_tetrio_data(CONFIG, T_CONFIG)


 - Loading Tetrio train...
 - Loading Tetrio test...
 - Loading Tetrio val...


In [8]:
i = 9819
print(val_loader.dataset.df.iloc[i]['playfield'])
print(val_loader.dataset.df.iloc[i]['playfield_next'])
print(val_loader.dataset.df.iloc[i]['immediate_garbage'])
val_loader.dataset.df.iloc[i]

GGGNGGGGGGGGGNGGGGGGGGGNGGGGGGGGGNGGGGGGGGGNGGGGGGGGGGGNGGGGGGGGGNGGGGGGGGGNGGGGGGGGGNGGGGGGGGGNGGGGGNGGGGGGGGGNGGGGGGGGGNGGGGGGGGGNGGGGGGGGGNGGGGGGGGGNGGGGGGGGGNGGGGGGGGGNGGGGGGGGGGGGGGGGGNGGGGGGGGGNJJISZLNTTIOOSSNNNNTIOOSNNNNNNI
GGGNGGGGGGGGGNGGGGGGGGGNGGGGGGGGGNGGGGGGGGGNGGGGGGGGGGGNGGGGGGGGGNGGGGGGGGGNGGGGGGGGGNGGGGGGGGGNGGGGGNGGGGGGGGGNGGGGGGGGGNGGGGGGGGGNGGGGGGGGGNGGGGGGGGGNGGGGGGGGGNGGGGGGGGGNGGGGGGGGGGGGGGGGGNGGGGGGGGGNOOSSNNZZTIOOSNNNNZNI
0


game_id                                                           2254
subframe                                                           155
playfield            GGGNGGGGGGGGGNGGGGGGGGGNGGGGGGGGGNGGGGGGGGGNGG...
playfield_next       GGGNGGGGGGGGGNGGGGGGGGGNGGGGGGGGGNGGGGGGGGGNGG...
real_current                                                         Z
placed                                                               Z
real_hold                                                            J
hold                                                                 J
next                                                    TJOIIJLSOZTZLJ
won                                                                  0
rating                                                    24921.703125
immediate_garbage                                                    0
btb                                                                  1
combo                                                                0
Name: 

```AssertionError at index 7633
AssertionError at index 9364
AssertionError at index 9819
AssertionError at index 9990
AssertionError at index 12078
AssertionError at index 13148
AssertionError at index 14938
AssertionError at index 21911
AssertionError at index 23840
AssertionError at index 24029
AssertionError at index 28171
AssertionError at index 28172
AssertionError at index 34473
AssertionError at index 34474
AssertionError at index 38943
AssertionError at index 49607
AssertionError at index 52696
AssertionError at index 53946
AssertionError at index 55175
```

In [137]:
for i in range(len(val_loader.dataset)):
    try:
        data = val_loader.dataset[i]
    except AssertionError:
        print(f"AssertionError at index {i}")

AssertionError at index 7633
AssertionError at index 9364
AssertionError at index 9819
AssertionError at index 9990
AssertionError at index 12078
AssertionError at index 13148
AssertionError at index 14938
AssertionError at index 21911
AssertionError at index 23840
AssertionError at index 24029
AssertionError at index 28171
AssertionError at index 28172
AssertionError at index 34473
AssertionError at index 34474
AssertionError at index 38943
AssertionError at index 49607
AssertionError at index 52696
AssertionError at index 53946
AssertionError at index 55175


KeyboardInterrupt: 

In [ ]:
for i in range(len(test_loader.dataset)):
    try:
        data = test_loader.dataset[i]
    except AssertionError:
        print(f"AssertionError at index {i}")

In [ ]:
for i in range(len(train_loader.dataset)):
    try:
        data = train_loader.dataset[i]
    except AssertionError:
        print(f"AssertionError at index {i}")

In [1]:
import sys, os
sys.path.insert(0, 'app/src')
sys.path.insert(0, 'app')
os.environ['PYGAME_HIDE_SUPPORT_PROMPT'] = '1'

import pandas as pd
import numpy as np
from multiprocessing import Pool, cpu_count
import time

from src.config import Configuration
from src.tetris import TetrisConfiguration, Tetris, Board, MoveSearcher
from src.data.tetrio import find_board_index

CONFIG = Configuration()
T_CONFIG = TetrisConfiguration(vanish_zone=5)

def check_row(idx):
    row = _df.iloc[idx]
    g = int(row['immediate_garbage'])
    
    game = Tetris(
        playfield=row['playfield'],
        next_pieces=row['next'],
        active_piece=row['placed'],
        hold_piece=row['hold'],
        vanish_zone=T_CONFIG.vanish_zone,
    )
    searcher = MoveSearcher(game, CONFIG, T_CONFIG)
    _, feats = searcher.get_all_features()
    
    ts = row['playfield_next'][g*10:]
    board = Board(10, 25, 5, False, ts)
    idx_match = find_board_index(board, feats['boards'])
    
    return idx, idx_match

def init_worker(df):
    global _df
    _df = df


df = pd.read_csv('../data/tetrio_val.csv')
n = len(df)
ncpu = int(cpu_count()*4/5)
print(f"Rows: {n}, workers: {ncpu}")

t0 = time.time()
fails = []
with Pool(processes=ncpu, initializer=init_worker, initargs=(df,)) as pool:
    for i, (idx, match) in enumerate(pool.imap_unordered(check_row, range(n), chunksize=100)):
        if match == -1:
            row = df.iloc[idx]
            fails.append((idx, row['game_id'], row['subframe'], row['placed']))
        if (i + 1) % 5000 == 0:
            elapsed = time.time() - t0
            rate = (i + 1) / elapsed
            print(f"  {i+1}/{n}  ({rate:.0f} rows/s)  fails: {len(fails)}")

elapsed = time.time() - t0
print(f"\nDone: {n} rows in {elapsed:.0f}s ({n/elapsed:.0f} rows/s)")
print(f"Failures: {len(fails)}")

if fails:
    for idx, gid, sf, placed in fails[:20]:
        print(f"  idx={idx} game={gid} frame={sf} placed={placed}")
    if len(fails) > 20:
        print(f"  ... and {len(fails)-20} more")

Rows: 752003, workers: 19


KeyboardInterrupt: 

# Show the data

In [ ]:
game_recod = raw_df.sort_values(by=['game_id', 'subframe'])

frame = 0

In [ ]:
from src.tetris import Tetris

playfield = game_recod.iloc[frame]['playfield']
playfield_next = game_recod.iloc[frame]['playfield_next']
next_pieces = game_recod.iloc[frame]['next']
current_piece = game_recod.iloc[frame]['real_current']
hold_piece = game_recod.iloc[frame]['real_hold']
# print(playfield)

game = Tetris(color_map=True, playfield=playfield, next_pieces=next_pieces, active_piece=current_piece, hold_piece=hold_piece)
game.print_state(include_vanish_zone=True)
frame += 1

In [ ]:
from src.tetris import MoveSearcher, Board, find_board_index

for i in range(1000):
    searcher = MoveSearcher(game, CONFIG, T_CONFIG)
    _, features = searcher.get_all_features()

import time 
start_time = time.time()
m = 10_000
for i in range(m):
    searcher = MoveSearcher(game, CONFIG, T_CONFIG)
    _, features = searcher.get_all_features()
print(f"Time for {m} iterations: {time.time() - start_time:.2f} seconds, {(time.time() - start_time)/m:.4f} seconds per iteration")

In [ ]:

searcher = MoveSearcher(game, CONFIG, T_CONFIG)
_, features = searcher.get_all_features()
boards_batch = features['boards']   # (128, 24, 10)

board = Board(game.width, game.height, game.vanish_zone, game.color_map, playfield_next)
board.print_board(include_vanish_zone=True)
idx = find_board_index(board, boards_batch)
# idx is the matching index in features, or -1

idx

In [ ]:
game = Tetris(color_map=True, playfield=playfield_next, next_pieces=next_pieces, active_piece=current_piece, hold_piece=hold_piece)
game.print_state()